# PyTorch Basics: Datasets and DataLoaders

This notebook covers:
- PyTorch Dataset class
- Built-in datasets (MNIST, CIFAR, ImageNet)
- Custom datasets
- DataLoader for batching
- Data transformations
- Data augmentation

## Why DataLoaders?

```
Raw Data → Dataset → DataLoader → Model
                    (batching,
                     shuffling,
                     parallel loading)
```

DataLoaders handle:
- **Batching**: Group samples together
- **Shuffling**: Randomize order
- **Parallel loading**: Use multiple CPU cores
- **Memory efficiency**: Load data on-demand


In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")


PyTorch version: 2.8.0
Torchvision version: 0.23.0


## 1. The Dataset Class

The `Dataset` class is an abstract class. You need to implement:
- `__len__()`: Returns the size of the dataset
- `__getitem__()`: Returns a sample at given index

```
class MyDataset(Dataset):
    def __init__(self):
        # Initialize data
        pass
    
    def __len__(self):
        # Return dataset size
        return len(self.data)
    
    def __getitem__(self, idx):
        # Return one sample
        return self.data[idx], self.label[idx]
```


In [2]:
# Simple custom dataset
class SimpleDataset(Dataset):
    def __init__(self, size=1000):
        # Generate random data
        self.data = torch.randn(size, 10)
        self.labels = torch.randint(0, 2, (size,))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Create dataset
dataset = SimpleDataset(size=100)
print(f"Dataset size: {len(dataset)}")
print(f"First sample: {dataset[0]}")
print(f"Data shape: {dataset[0][0].shape}")
print(f"Label: {dataset[0][1]}")


Dataset size: 100
First sample: (tensor([ 1.8195,  1.4306,  2.4056, -0.7773, -1.3438, -0.5649, -1.5310, -1.2556,
        -0.6181, -2.6560]), tensor(0))
Data shape: torch.Size([10])
Label: 0


## 2. DataLoader for Batching

DataLoader wraps a Dataset and provides batching, shuffling, and parallel loading.


In [3]:
# Create DataLoader
dataloader = DataLoader(
    dataset, 
    batch_size=10,
    shuffle=True,
    num_workers=0  # Set to 0 for compatibility
)

print(f"Number of batches: {len(dataloader)}")
print()

# Iterate through batches
for batch_idx, (data, labels) in enumerate(dataloader):
    print(f"Batch {batch_idx}:")
    print(f"  Data shape: {data.shape}")
    print(f"  Labels shape: {labels.shape}")
    
    if batch_idx == 2:  # Show first 3 batches
        break


Number of batches: 10

Batch 0:
  Data shape: torch.Size([10, 10])
  Labels shape: torch.Size([10])
Batch 1:
  Data shape: torch.Size([10, 10])
  Labels shape: torch.Size([10])
Batch 2:
  Data shape: torch.Size([10, 10])
  Labels shape: torch.Size([10])


## 3. Built-in Datasets: MNIST

PyTorch provides many built-in datasets through `torchvision.datasets`.


In [4]:
# Download and load MNIST
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print()

# Inspect a sample
image, label = train_dataset[0]
print(f"Image shape: {image.shape}")  # (channels, height, width)
print(f"Label: {label}")
print(f"Image value range: [{image.min():.3f}, {image.max():.3f}]")


Training samples: 60000
Test samples: 10000

Image shape: torch.Size([1, 28, 28])
Label: 5
Image value range: [0.000, 1.000]


## 4. Data Transformations

Transforms preprocess data before feeding to the model.

Common transforms:
- `ToTensor()`: Convert PIL/numpy to tensor
- `Normalize()`: Normalize with mean and std
- `Resize()`: Resize images
- `RandomCrop()`: Random cropping
- `RandomHorizontalFlip()`: Random flipping


In [5]:
# Compose multiple transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

image, label = dataset[0]
print(f"After normalization:")
print(f"Mean: {image.mean():.4f}")
print(f"Std: {image.std():.4f}")
print(f"Value range: [{image.min():.3f}, {image.max():.3f}]")


After normalization:
Mean: 0.0227
Std: 1.0144
Value range: [-0.424, 2.821]


## 5. Complete Training Pipeline Example


In [6]:
# Setup
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
print()

# Peek at one batch
data_iter = iter(train_loader)
images, labels = next(data_iter)
print(f"Batch shape: {images.shape}")  # (batch_size, channels, height, width)
print(f"Labels shape: {labels.shape}")
print(f"Labels: {labels[:10]}")


Training batches: 938
Test batches: 10

Batch shape: torch.Size([64, 1, 28, 28])
Labels shape: torch.Size([64])
Labels: tensor([2, 3, 5, 9, 2, 6, 9, 1, 8, 3])


## 📝 Summary

✅ Created custom datasets  
✅ Used built-in datasets  
✅ Applied data transformations  
✅ Created DataLoaders for batching  
✅ Built complete data pipeline  

### Next Steps

- **Next Tutorial**: `04_neural_networks.ipynb`
- **Practice**: Create a custom dataset for your own data
- **Challenge**: Implement data augmentation for CIFAR-10

### Additional Resources

- [PyTorch Data Loading Tutorial](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html)
- [Torchvision Datasets](https://pytorch.org/vision/stable/datasets.html)
- [Torchvision Transforms](https://pytorch.org/vision/stable/transforms.html)


## 🎯 Practice Exercises


In [7]:
# Exercise 1: Create a dataset for regression (X, y pairs)
# Your code here:


# Exercise 2: Implement a custom transform that adds noise to images
# Your code here:


# Exercise 3: Create a DataLoader with custom collate_fn
# Your code here:


# Exercise 4: Load CIFAR-10 and apply data augmentation
# Your code here:
